In [ ]:
import	pandas					as	pd
import	plotly.graph_objects	as	go
import	plotly.express			as	px
import	numpy					as	np
rng = np.random.default_rng()
df_btc: pd.DataFrame = pd.read_parquet('data/IBIT_data.parquet')
df_eth: pd.DataFrame = pd.read_parquet('data/ETHA_data.parquet')
for df in [df_btc, df_eth]:
	df['timestamp'] = pd.to_datetime(df['timestamp'])
	df['date'] = df['timestamp'].dt.normalize()
	df = df[df['date'] >= pd.Timestamp('2026-01-01', tz = 'America/New_York')]
	df['time'] = df['timestamp'] - df['timestamp'].dt.normalize()
	cols = ['timestamp', 'date', 'time'] + [c for c in df.columns if c not in ['timestamp', 'date', 'time']]
	df = df[cols]
common = set(df_btc.timestamp) & set(df_eth.timestamp)
df_btc = df_btc[df_btc.timestamp.isin(common)].reset_index()
df_eth = df_eth[df_eth.timestamp.isin(common)].reset_index()

In [ ]:
# so we know now there is an approximate linear relation between the two but the intercept tends to drift, luckily slope doesn't
# px.line(x = df_eth.vwap, y = df_btc.vwap)

In [ ]:
import	scipy.stats	as	stat
r, p = stat.pearsonr(df_eth.vwap, df_btc.vwap)
r, p

In [ ]:
a, b = np.polyfit(df_eth.vwap, df_btc.vwap, deg = 1)
a, b

In [ ]:
return_eth: pd.Series = np.log(df_eth.vwap.shift(1) / df_eth.vwap).dropna()
return_btc: pd.Series = np.log(df_btc.vwap.shift(1) / df_btc.vwap).dropna()

In [ ]:
# returns are always, always gaussian-connected, i.e. (r_btc, r_eth) is sampled from a Gaussian, even when r_btc and r_eth are t-distributed individually.
px.scatter(x = return_btc, y = return_eth)

In [ ]:
r_r, p_r = stat.pearsonr(return_eth.values, return_btc.values)
a_r, b_r = np.polyfit(return_eth.values, return_btc.values, deg = 1)
r_r, p_r, a_r, b_r

In [ ]:
data = pd.Series(return_btc.values - a_r * return_eth.values - b_r)

In [ ]:
t_param = stat.t.fit(data)
lap_param = stat.laplace.fit(data)
print(t_param, lap_param)
t_param_btc = stat.t.fit(return_btc)
lap_param_btc = stat.laplace.fit(return_btc)
print(t_param_btc, lap_param_btc)
t_param_eth = stat.t.fit(return_eth)
lap_param_eth = stat.laplace.fit(return_eth)
print(t_param_eth, lap_param_eth)

In [ ]:
print(stat.kurtosis((return_btc.values - a_r * return_eth.values - b_r)))
print(stat.skew((return_btc.values - a_r * return_eth.values - b_r)))
returns_fig = go.Figure()
x_axis_returns: np.ndarray = np.linspace(data.min(), data.max(), 2000)
t_fitted_pdf = stat.t.pdf(x_axis_returns, *t_param)
returns_fig.add_trace(go.Histogram(x = (return_btc.values - a_r * return_eth.values - b_r), histnorm = 'probability density', name = 'Empirical', opacity = 0.5))
returns_fig.add_trace(go.Scatter(x = x_axis_returns, y = t_fitted_pdf, mode = 'lines', name = 'Fitted t', line = dict(color = 'red')))
# returns_fig.update_yaxes(type='log')
# px.histogram(x = return_eth)

In [ ]:
print(stat.kurtosis(return_btc.values))
print(stat.skew(return_btc))
returns_fig = go.Figure()
x_axis_returns_btc: np.ndarray = np.linspace(return_btc.min(), return_btc.max(), 2000)
t_fitted_pdf_btc = stat.t.pdf(x_axis_returns_btc, *t_param_btc)
returns_fig.add_trace(go.Histogram(x = return_btc, histnorm = 'probability density', name = 'Empirical', opacity = 0.5))
returns_fig.add_trace(go.Scatter(x = x_axis_returns_btc, y = t_fitted_pdf_btc, mode = 'lines', name = 'Fitted t', line = dict(color = 'red')))
# returns_fig.update_yaxes(type='log')
# px.histogram(x = return_eth)

In [ ]:
print(stat.kurtosis(return_eth.values))
print(stat.skew(return_eth))
returns_fig = go.Figure()
x_axis_returns_eth: np.ndarray = np.linspace(return_eth.min(), return_eth.max(), 2000)
t_fitted_pdf_eth = stat.t.pdf(x_axis_returns_eth, *t_param_eth)
returns_fig.add_trace(go.Histogram(x = return_eth, histnorm = 'probability density', name = 'Empirical', opacity = 0.5))
returns_fig.add_trace(go.Scatter(x = x_axis_returns_eth, y = t_fitted_pdf_eth, mode = 'lines', name = 'Fitted t', line = dict(color = 'red')))
# returns_fig.update_yaxes(type='log')
# px.histogram(x = return_eth)

In [ ]:
from	statsmodels.graphics.tsaplots	import	plot_acf
residuals: pd.Series = (df_btc.vwap - a * df_eth.vwap - b)
residuals_diff: pd.Series = np.log(residuals.shift(1)/residuals).dropna()
acf_fig = plot_acf(residuals_diff, lags = 20)

In [ ]:
ks_stat, ks_p = stat.kstest(data, 't', args = t_param)
ks_stat, ks_p

In [ ]:
# so we can currently model p(return) as a t-distribution with 2 degrees of freedom, cooked but still doable
# is it time to do p(return|previous return?)
arr1 = stat.t.rvs((t_param[0],), loc = t_param[1], scale = t_param[2], size = len(return_btc.values))
arr2 = stat.t.rvs((t_param[0],), loc = t_param[1], scale = t_param[2], size = len(return_btc.values))
random_vars = px.scatter(x = arr1, y = arr2)
random_vars.show()
# as you can see we're a bit cooked, in reality the variance around the middle is often LARGER than naive independence for |t_1 - t_2| ≤ 10
# so currently from eyeballing p(r_t) and p(r_{t+k}) for |k| ≥ 10 (at LEAST), MI ~ 0.
# this is good! it means that p(r_t|r_{t-1}...) = p(r_t|r_{t-1}, r_{t-2}, ...)
px.scatter(x = (return_btc.values), y = (return_btc.shift(1).values))